In [1]:
!pip install -q imbalanced-learn lightgbm xgboost

In [2]:
import os
import sys
import math
import pickle
import warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               StackingClassifier, VotingClassifier,
                               ExtraTreesClassifier, HistGradientBoostingClassifier,
                               BaggingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
from sklearn.feature_selection import SelectKBest, f_classif

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')


In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/projectPG"
os.makedirs(BASE_DIR, exist_ok=True)
os.chdir(BASE_DIR)

Mounted at /content/drive


In [4]:
DATASET_DIR  = os.path.join(BASE_DIR, "ReadingConfidenceDataset-main")
AUDIO_DIR    = os.path.join(DATASET_DIR, "audios")
RATINGS_CSV  = os.path.join(DATASET_DIR, "ratings.csv")

# ── Trained breath-detection checkpoint (Conformer + BiLSTM) ──────────
MODEL_PATH    = os.path.join(BASE_DIR, "3breath_model_best.pt")
BEST_CLF_PATH = os.path.join(BASE_DIR, "module3_best_model.pkl")


THRESHOLD     = 0.65

SAMPLE_RATE    = 16_000
SR             = SAMPLE_RATE
FRAME_LEN      = int(SAMPLE_RATE * 0.025)
HOP_LEN        = int(SAMPLE_RATE * 0.010)
HOP_LEN_SEC    = 0.010
N_FFT          = FRAME_LEN
HOP            = HOP_LEN
N_MELS         = 64
MODEL_DIM      = 128
NUM_HEADS      = 4
NUM_CONFORMERS = 3
NUM_BILSTM     = 1
DROPOUT        = 0.2
FEAT_DIM       = N_MELS + 2

TEST_SIZE     = 0.20
RANDOM_STATE  = 42


In [5]:
# ── Building block 1/3: Convolution Module
class ConvolutionModule(nn.Module):
    def __init__(self, channels, kernel_size=31):
        super().__init__()
        self.ln  = nn.LayerNorm(channels)
        self.pw1 = nn.Conv1d(channels, 2*channels, 1)
        self.glu = nn.GLU(dim=1)
        self.dw  = nn.Conv1d(channels, channels, kernel_size,
                             padding=kernel_size//2, groups=channels)
        self.bn  = nn.BatchNorm1d(channels)
        self.act = nn.SiLU()
        self.pw2 = nn.Conv1d(channels, channels, 1)
        self.drop= nn.Dropout(DROPOUT)

    def forward(self, x):
        r = x
        x = self.ln(x).transpose(1,2)
        x = self.glu(self.pw1(x))
        x = self.act(self.bn(self.dw(x)))
        x = self.drop(self.pw2(x)).transpose(1,2)
        return x + r


In [6]:
# ── Building block 2/3: Feed-Forward Module ──────────────────────
class FeedForwardModule(nn.Module):
    def __init__(self, dim, expansion=4):
        super().__init__()
        self.ln  = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim*expansion)
        self.act = nn.SiLU()
        self.d1  = nn.Dropout(DROPOUT)
        self.fc2 = nn.Linear(dim*expansion, dim)
        self.d2  = nn.Dropout(DROPOUT)

    def forward(self, x):
        return x + 0.5*self.d2(self.fc2(self.d1(self.act(self.fc1(self.ln(x))))))


In [7]:

class ConformerBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.ff1    = FeedForwardModule(dim)
        self.ln     = nn.LayerNorm(dim)
        self.attn   = nn.MultiheadAttention(dim, num_heads, dropout=DROPOUT, batch_first=True)
        self.drop   = nn.Dropout(DROPOUT)
        self.conv   = ConvolutionModule(dim)
        self.ff2    = FeedForwardModule(dim)
        self.ln_out = nn.LayerNorm(dim)

    def forward(self, x, key_padding_mask=None):
        x = self.ff1(x)
        r = x
        x_ln = self.ln(x)
        x, _ = self.attn(x_ln, x_ln, x_ln, key_padding_mask=key_padding_mask)
        x = self.conv(self.ff2(self.ln_out(r + self.drop(x))))
        return x


class ConformerBiLSTM(nn.Module):

    def __init__(self):
        super().__init__()
        self.subsample = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU()
        )
        sub_feat = 32 * math.ceil(math.ceil(FEAT_DIM/2)/2)
        self.proj       = nn.Linear(sub_feat, MODEL_DIM)
        self.drop       = nn.Dropout(DROPOUT)
        self.conformers = nn.ModuleList([
            ConformerBlock(MODEL_DIM, NUM_HEADS)
            for _ in range(NUM_CONFORMERS)
        ])
        self.upsample = nn.Sequential(
            nn.ConvTranspose1d(MODEL_DIM, MODEL_DIM, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose1d(MODEL_DIM, MODEL_DIM, 4, stride=2, padding=1), nn.ReLU()
        )
        self.bilstm = nn.LSTM(
            MODEL_DIM, MODEL_DIM//2,
            num_layers=NUM_BILSTM,
            batch_first=True, bidirectional=True,
            dropout=DROPOUT if NUM_BILSTM > 1 else 0.0
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(MODEL_DIM),
            nn.Linear(MODEL_DIM, 1)
        )

    def forward(self, x, mask=None):
        B, T, Fdim = x.shape
        x_2d = self.subsample(x.unsqueeze(1))
        B2, C2, T2, F2 = x_2d.shape
        x_flat = self.drop(self.proj(
            x_2d.permute(0,2,1,3).contiguous().view(B2, T2, C2*F2)
        ))
        pad_mask = (~mask[:, ::4][:, :T2]) if mask is not None else None
        for blk in self.conformers:
            x_flat = blk(x_flat, key_padding_mask=pad_mask)
        x_up = self.upsample(x_flat.transpose(1,2)).transpose(1,2)
        T_up = x_up.shape[1]
        if T_up >= T:
            x_up = x_up[:, :T, :]
        else:
            pad = torch.zeros(B, T-T_up, x_up.shape[2], device=x_up.device)
            x_up = torch.cat([x_up, pad], dim=1)
        x_lstm, _ = self.bilstm(x_up)
        return self.classifier(x_lstm).squeeze(-1)  # (B, T) logits


In [8]:
# STEP 1 — Preprocessing + Feature Extraction
def extract_model_features(waveform, sr=SAMPLE_RATE):

    if sr != SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=sr, target_sr=SAMPLE_RATE)
    mel = librosa.feature.melspectrogram(
        y=waveform, sr=SAMPLE_RATE,
        n_fft=FRAME_LEN, hop_length=HOP_LEN,
        win_length=FRAME_LEN, n_mels=N_MELS, power=2.0
    )
    mel_db = librosa.power_to_db(mel + 1e-9, ref=1.0).T
    zcr = librosa.feature.zero_crossing_rate(
        waveform, frame_length=FRAME_LEN, hop_length=HOP_LEN
    ).T
    vms = np.var(mel_db, axis=1, keepdims=True)
    T = min(mel_db.shape[0], zcr.shape[0], vms.shape[0])
    raw_feat = np.concatenate([mel_db[:T], zcr[:T], vms[:T]], axis=1).astype(np.float32)
    zcr_1d   = zcr[:T, 0].astype(np.float32)
    return raw_feat, zcr_1d


def normalize_features(features):
    mean = features.mean(axis=0, keepdims=True)
    std  = features.std(axis=0,  keepdims=True) + 1e-8
    return (features - mean) / std


def extract_features(audio_path):

    wav, _ = librosa.load(audio_path, sr=SR, mono=True)
    mx = np.max(np.abs(wav))
    if mx > 0:
        wav = wav / mx

    raw_feat, zcr_raw = extract_model_features(wav, SR)
    feat_norm = normalize_features(raw_feat)
    duration_sec = len(wav) / SR
    return feat_norm, zcr_raw, duration_sec, wav


In [9]:
# STEP 2 — Run Model → frame-level probs
def run_model(feat_norm, model, device):
    x    = torch.tensor(feat_norm, dtype=torch.float32).unsqueeze(0).to(device)   # (1, T, FEAT_DIM)
    mask = torch.ones(1, x.shape[1], dtype=torch.bool).to(device)
    with torch.no_grad():
        logits = model(x, mask)
        probs  = torch.sigmoid(logits)
    return probs.squeeze(0).cpu().numpy()


In [10]:
# STEP 3 — Extract Breathing Features from frame decisions
def get_breath_segments(decisions, hop_sec=0.01):
    segs = []; in_b = False; s = 0
    for i, d in enumerate(decisions):
        if d == 1 and not in_b:
            in_b = True; s = i
        elif d == 0 and in_b:
            in_b = False; segs.append((s * hop_sec, i * hop_sec))
    if in_b:
        segs.append((s * hop_sec, len(decisions) * hop_sec))
    return segs

def breathing_features(probs, duration_sec, threshold):
    #8 breathing features
    decisions = (probs >= threshold).astype(int)
    segs      = get_breath_segments(decisions)
    n         = len(segs)

    if n == 0:
        return np.zeros(8, dtype=np.float32)

    rate     = (n / duration_sec) * 60.0
    durs     = [e - s for s, e in segs]
    mean_dur = float(np.mean(durs))
    std_dur  = float(np.std(durs))

    if n > 1:
        starts = [s for s, _ in segs]
        reg    = float(np.std(np.diff(starts)))
    else:
        reg = 0.0

    ratio  = float(decisions.sum() / len(decisions))
    mean_p = float(probs.mean())
    max_p  = float(probs.max())

    return np.array([rate, mean_dur, std_dur, reg,
                     ratio, mean_p, max_p, n], dtype=np.float32)

def audio_features(wav, zcr, duration_sec):

    # RMS energy
    rms = librosa.feature.rms(y=wav, frame_length=N_FFT, hop_length=HOP)[0]
    rms_mean = float(np.mean(rms))
    rms_std  = float(np.std(rms))
    rms_max  = float(np.max(rms))

    # ZCR from pre-computed array
    zcr_mean = float(np.mean(zcr))
    zcr_std  = float(np.std(zcr))

    # Spectral centroid
    sc = librosa.feature.spectral_centroid(y=wav, sr=SR, n_fft=N_FFT, hop_length=HOP)[0]
    sc_mean = float(np.mean(sc))
    sc_std  = float(np.std(sc))

    # Spectral bandwidth
    sb = librosa.feature.spectral_bandwidth(y=wav, sr=SR, n_fft=N_FFT, hop_length=HOP)[0]
    sb_mean = float(np.mean(sb))

    # Spectral rolloff
    sr_feat = librosa.feature.spectral_rolloff(y=wav, sr=SR, n_fft=N_FFT, hop_length=HOP)[0]
    sr_mean = float(np.mean(sr_feat))

    # MFCCs (first 3)
    mfcc = librosa.feature.mfcc(y=wav, sr=SR, n_mfcc=5, hop_length=HOP)
    mfcc1 = float(np.mean(mfcc[0]))
    mfcc2 = float(np.mean(mfcc[1]))
    mfcc3 = float(np.mean(mfcc[2]))

    # Speech ratio
    threshold_e = rms_mean * 0.5
    speech_ratio = float(np.mean(rms > threshold_e))

    return np.array([
        rms_mean, rms_std, rms_max,
        zcr_mean, zcr_std,
        sc_mean, sc_std,
        sb_mean, sr_mean,
        mfcc1, mfcc2, mfcc3,
        speech_ratio
    ], dtype=np.float32)

BREATH_FEATURE_NAMES = [
    "breathing_rate", "mean_breath_dur", "std_breath_dur",
    "pause_regularity", "breath_ratio",
    "mean_prob", "max_prob", "num_segments"
]

AUDIO_FEATURE_NAMES = [
    "rms_mean", "rms_std", "rms_max",
    "zcr_mean", "zcr_std",
    "spec_centroid_mean", "spec_centroid_std",
    "spec_bw_mean", "spec_rolloff_mean",
    "mfcc1_mean", "mfcc2_mean", "mfcc3_mean",
    "speech_ratio"
]

FEATURE_NAMES = BREATH_FEATURE_NAMES + AUDIO_FEATURE_NAMES  # 21 total


In [11]:
# STEP 4 — Method 1: Formula-Based
def formula_predict(feats):
    rate, _, _, reg, ratio = feats[0], feats[1], feats[2], feats[3], feats[4]

    if 12 <= rate <= 20:   rate_s = 1.0
    elif rate < 12:        rate_s = max(0.0, rate / 12.0)
    else:                  rate_s = max(0.0, 1.0 - (rate - 20) / 20.0)

    reg_s = float(np.exp(-reg))

    if 0.05 <= ratio <= 0.30:  ratio_s = 1.0
    elif ratio < 0.05:         ratio_s = ratio / 0.05
    else:                      ratio_s = max(0.0, 1.0 - (ratio - 0.30) / 0.70)

    score = 0.35 * rate_s + 0.45 * reg_s + 0.20 * ratio_s

    if score >= 0.65:   return 1
    elif score >= 0.40: return 0
    else:               return -1

In [12]:
# STEP 5 — Load dataset + run inference on all files
def load_dataset():
    for sep in ['\t', ',', r'\s+']:
        try:
            df = pd.read_csv(RATINGS_CSV, header=None, sep=sep,
                             names=['filename', 'label', 'split'],
                             engine='python')
            df['label'] = pd.to_numeric(df['label'], errors='coerce')
            df = df.dropna(subset=['label'])
            df['label'] = df['label'].astype(int)
            df['filename'] = df['filename'].astype(str).str.strip()
            if len(df) > 5:
                return df
        except Exception:
            continue
    raise ValueError(f"Could not parse {RATINGS_CSV}")

def build_feature_matrix(df, model, device):

    X, y, filenames = [], [], []
    print(f"\nProcessing {len(df)} audio files...")
    failed = 0

    for i, row in df.iterrows():
        fname = str(row['filename']).strip()
        try:
            label = int(row['label'])
        except (ValueError, TypeError):
            failed += 1
            continue

        candidates = [
            os.path.join(AUDIO_DIR, fname + ".wav"),
            os.path.join(AUDIO_DIR, fname),
        ]
        path = next((p for p in candidates if os.path.exists(p)), None)
        if path is None:
            failed += 1
            continue

        try:
            feat_norm, zcr_raw, dur, wav = extract_features(path)
            probs   = run_model(feat_norm, model, device)
            b_feats = breathing_features(probs, dur, THRESHOLD)
            a_feats = audio_features(wav, zcr_raw, dur)
            feats   = np.concatenate([b_feats, a_feats])
            X.append(feats)
            y.append(label)
            filenames.append(fname)
            if len(X) % 50 == 0:
                print(f"  {len(X)} processed...")
        except Exception as e:
            failed += 1
            if failed <= 3:
                print(f"  [skip] {fname}: {e}")

    print(f"  Done — {len(X)} files processed, {failed} skipped")
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), filenames


In [13]:
# STEP 6 — Train+Val / Test
def split_data(X, y_int, filenames):

    indices = np.arange(len(X))

    idx_tv, idx_test = train_test_split(
        indices,
        test_size=TEST_SIZE,
        stratify=y_int,
        random_state=RANDOM_STATE
    )

    X_trainval  = X[idx_tv]
    X_test      = X[idx_test]
    y_trainval  = y_int[idx_tv]
    y_test      = y_int[idx_test]
    fnames_tv   = [filenames[i] for i in idx_tv]
    fnames_test = [filenames[i] for i in idx_test]

    print(f"\n{'='*55}")
    print("  Data Split")
    print(f"{'='*55}")
    print(f"  Total samples       : {len(X)}")
    print(f"  Train+Val ({int((1-TEST_SIZE)*100)}%)  : {len(X_trainval)} samples")
    print(f"  Test Set  ({int(TEST_SIZE*100)}%)   : {len(X_test)} samples")
    print(f"\n  Class distribution in Train+Val:")
    for cls, cnt in zip(*np.unique(y_trainval, return_counts=True)):
        print(f"    Class {cls}: {cnt} samples")
    print(f"\n  Class distribution in Test Set:")
    for cls, cnt in zip(*np.unique(y_test, return_counts=True)):
        print(f"    Class {cls}: {cnt} samples")

    return X_trainval, X_test, y_trainval, y_test, fnames_tv, fnames_test

In [14]:
# STEP 7 — SMOTE
def label_to_int(y):
    return np.array([v + 1 for v in y], dtype=np.int32)

def int_to_label(y):
    return np.array([v - 1 for v in y], dtype=np.int32)

LABEL_NAMES = ['Low (-1)', 'Medium (0)', 'High (1)']

def smote_resample(X, y_int):

    if not HAS_SMOTE:
        return X, y_int
    counts = np.bincount(y_int)
    min_count = counts.min()
    if min_count < 2:
        return X, y_int
    try:
        k = min(5, min_count - 1)
        sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=k)
        X_res, y_res = sm.fit_resample(X, y_int)
        print(f"  SMOTE (on Train+Val only): {len(X)} → {len(X_res)} samples")
        return X_res, y_res
    except Exception as e:
        print(f"  SMOTE failed: {e}")
        return X, y_int

In [15]:
# STEP 8 — Evaluate classifiers
def evaluate_classifier(name, clf, X_trainval, y_trainval, cv=5):

    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    scoring = ['accuracy', 'f1_macro', 'f1_weighted']
    scores  = cross_validate(clf, X_trainval, y_trainval, cv=skf, scoring=scoring,
                             return_train_score=False)
    return {
        'Accuracy':     scores['test_accuracy'].mean(),
        'Accuracy_std': scores['test_accuracy'].std(),
        'F1_macro':     scores['test_f1_macro'].mean(),
        'F1_macro_std': scores['test_f1_macro'].std(),
        'F1_weighted':  scores['test_f1_weighted'].mean(),
        'F1_w_std':     scores['test_f1_weighted'].std(),
    }

def full_report(name, clf, X_trainval, y_trainval):
    clf.fit(X_trainval, y_trainval)
    y_pred = clf.predict(X_trainval)
    print(f"\n{'='*55}")
    print(f"  {name}  — Detailed Report (Train+Val set)")
    print(f"{'='*55}")
    print(classification_report(y_trainval, y_pred,
                                 target_names=LABEL_NAMES, zero_division=0))
    return confusion_matrix(y_trainval, y_pred)

def final_test_evaluation(name, clf, X_trainval_aug, y_trainval_aug,
                          X_test, y_test, fnames_test):

    clf.fit(X_trainval_aug, y_trainval_aug)
    y_pred_test = clf.predict(X_test)

    test_acc  = accuracy_score(y_test, y_pred_test)
    test_f1m  = f1_score(y_test, y_pred_test, average='macro',    zero_division=0)
    test_f1w  = f1_score(y_test, y_pred_test, average='weighted', zero_division=0)
    cm_test   = confusion_matrix(y_test, y_pred_test)

    print(f"\n{'='*55}")
    print(f"  Final Result on Test Set — {name}")
    print(f"{'='*55}")
    print(f"  Test Accuracy   : {test_acc:.4f}")
    print(f"  Test F1 Macro   : {test_f1m:.4f}")
    print(f"  Test F1 Weighted: {test_f1w:.4f}")
    print("\n  Classification Report (Test Set):")
    print(classification_report(y_test, y_pred_test,
                                 target_names=LABEL_NAMES, zero_division=0))

    return test_acc, test_f1m, test_f1w, cm_test


In [16]:
# STEP 9 — Plots
def plot_confusion_matrix(cm, name, ax):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

def plot_results(results_df, save_path='module3_results.png'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Module III — Confidence Classification Results (CV on Train+Val)',
                 fontsize=14, fontweight='bold')

    methods = results_df['Method']
    x = np.arange(len(methods))
    w = 0.28

    ax = axes[0]
    ax.bar(x - w, results_df['Accuracy'],    w, label='Accuracy',    color="#42092B", alpha=0.85)
    ax.bar(x,     results_df['F1_macro'],    w, label='F1 Macro',    color="#AD5AA6", alpha=0.85)
    ax.bar(x + w, results_df['F1_weighted'], w, label='F1 Weighted', color="#3BC9D6", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=25, ha='right', fontsize=8)
    ax.set_ylim(0, 1.0); ax.set_ylabel('Score')
    ax.set_title('Cross-Validation Scores on Train+Val (5-fold)'); ax.legend(fontsize=9)
    ax.axhline(0.7, color='red', linestyle='--', alpha=0.5, label='0.7 target')
    ax.grid(axis='y', alpha=0.3)

    ax2 = axes[1]
    sorted_df = results_df.sort_values('Accuracy', ascending=True)
    colors = ['#42092B' if v < 0.5 else '#AD5AA6' if v < 0.7 else '#3BC9D6'
              for v in sorted_df['Accuracy']]
    bars = ax2.barh(sorted_df['Method'], sorted_df['Accuracy'],
                    color=colors, alpha=0.85)
    ax2.set_xlim(0, 1.0)
    ax2.set_title('Accuracy Ranking (CV on Train+Val)')
    ax2.set_xlabel('Accuracy')
    ax2.axvline(0.7, color='red', linestyle='--', alpha=0.5)
    for bar, val in zip(bars, sorted_df['Accuracy']):
        ax2.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                 f'{val:.3f}', va='center', fontsize=9)
    ax2.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"\nPlot saved to {save_path}")

In [19]:
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    # ── Load Model ──
    print("\n[1] Loading trained ConformerBiLSTM checkpoint...")
    model = ConformerBiLSTM().to(device)
    if not os.path.exists(MODEL_PATH):
        print(f"ERROR: {MODEL_PATH} not found.")
        print("       ارفع ملف الشيكبوينت (3breath_model_best.pt) لنفس مجلد المشروع بجوجل درايف.")
        sys.exit(1)
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    if 'val_metrics' in checkpoint:
        print(f"    Model loaded  (checkpoint epoch={checkpoint.get('epoch','?')}, "
              f"val F1={checkpoint['val_metrics'].get('f1', float('nan')):.4f})")
    else:
        print("    Model loaded ")

    # ── Load Dataset ──
    print("\n[2] Loading ratings.csv...")
    df = load_dataset()
    print(f"    Total entries: {len(df)}")
    print(f"    Label distribution:\n{df['label'].value_counts().to_string()}")

    # ── Build Feature Matrix ──
    print("\n[3] Extracting features from all audio files...")
    X, y, filenames = build_feature_matrix(df, model, device)
    print(f"\n    Feature matrix shape: {X.shape}")
    print(f"    Labels distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

    if len(X) == 0:
        print("ERROR: No files processed. Check AUDIO_DIR path.")
        sys.exit(1)

    print("\n    Feature Statistics:")
    feat_df = pd.DataFrame(X, columns=FEATURE_NAMES)
    print(feat_df.describe().round(3).to_string())

    y_int = label_to_int(y)


    X_trainval, X_test, y_trainval, y_test, fnames_tv, fnames_test = split_data(
        X, y_int, filenames
    )

    print("\n[4] Applying SMOTE on Train+Val only (prevents data leakage)...")
    X_aug, y_aug = smote_resample(X_trainval, y_trainval)

    print("\n" + "="*55)
    print("  METHOD 1: Formula-Based (as in Report)")
    print("  Evaluated on: Train+Val")
    print("="*55)
    y_trainval_orig = int_to_label(y_trainval)
    formula_preds_tv = np.array([formula_predict(x) for x in X_trainval])
    formula_acc_tv   = accuracy_score(y_trainval_orig, formula_preds_tv)
    formula_f1m_tv   = f1_score(y_trainval_orig, formula_preds_tv, average='macro',    zero_division=0)
    formula_f1w_tv   = f1_score(y_trainval_orig, formula_preds_tv, average='weighted', zero_division=0)
    print(f"  Accuracy (Train+Val)   : {formula_acc_tv:.4f}")
    print(f"  F1 Macro (Train+Val)   : {formula_f1m_tv:.4f}")
    print(f"  F1 Weighted (Train+Val): {formula_f1w_tv:.4f}")
    cm_formula = confusion_matrix(y_trainval_orig, formula_preds_tv)

    # ── Classifiers ──
    classifiers = {
        'Random Forest': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', RandomForestClassifier(
                n_estimators=300, max_depth=10, min_samples_leaf=2,
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
        ]),
        'SVM (RBF)': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', SVC(kernel='rbf', C=10.0, gamma='scale',
                        class_weight='balanced', probability=True, random_state=RANDOM_STATE))
        ]),
        'Gradient Boosting': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', GradientBoostingClassifier(
                n_estimators=300, learning_rate=0.05,
                max_depth=4, subsample=0.8, random_state=RANDOM_STATE))
        ]),
        'MLP': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', MLPClassifier(
                hidden_layer_sizes=(256, 128, 64), activation='relu',
                max_iter=800, early_stopping=True, random_state=RANDOM_STATE,
                learning_rate_init=0.001, alpha=0.001))
        ]),
        'KNN': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', KNeighborsClassifier(n_neighbors=5, metric='euclidean', weights='distance'))
        ]),
        'Logistic Regression': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', LogisticRegression(C=1.0, class_weight='balanced',
                                        max_iter=1000, random_state=RANDOM_STATE))
        ]),


        # Method 9: Extra Trees
        'Extra Trees': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', ExtraTreesClassifier(
                n_estimators=300, max_depth=10, min_samples_leaf=2,
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
        ]),

        # Method 10: HistGradientBoosting — Boosting
        'HistGradientBoosting': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', HistGradientBoostingClassifier(
                max_iter=300, learning_rate=0.05, max_depth=5,
                class_weight='balanced', random_state=RANDOM_STATE))
        ]),

        # Method 11: QDA
        'QDA': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', QuadraticDiscriminantAnalysis(reg_param=0.1))
        ]),

        # Method 12: Gaussian Naive Bayes
        'Gaussian NB': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', GaussianNB(var_smoothing=1e-8))
        ]),

        # Method 13: Bagging
        'Bagging': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', BaggingClassifier(
                estimator=DecisionTreeClassifier(max_depth=8),
                n_estimators=200, max_samples=0.8, max_features=0.8,
                random_state=RANDOM_STATE, n_jobs=-1))
        ]),
    }

    if HAS_XGBOOST:
        classifiers['XGBoost'] = Pipeline([
            ('scaler', RobustScaler()),
            ('clf', XGBClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=5,
                subsample=0.8, colsample_bytree=0.8,
                eval_metric='mlogloss', random_state=RANDOM_STATE, verbosity=0))
        ])

    if HAS_LIGHTGBM:
        classifiers['LightGBM'] = Pipeline([
            ('scaler', RobustScaler()),
            ('clf', LGBMClassifier(
                n_estimators=300, learning_rate=0.05,
                class_weight='balanced', random_state=RANDOM_STATE, verbose=-1))
        ])


    base_estimators = [
        ('svm',  SVC(kernel='rbf', C=10.0, gamma='scale',
                     class_weight='balanced', probability=True, random_state=RANDOM_STATE)),

        ('et',   ExtraTreesClassifier(
                     n_estimators=200, max_depth=8, class_weight='balanced',
                     random_state=RANDOM_STATE, n_jobs=-1)),

        ('hgb',  HistGradientBoostingClassifier(
                     max_iter=200, learning_rate=0.05, max_depth=5,
                     class_weight='balanced', random_state=RANDOM_STATE)),

        ('qda',  QuadraticDiscriminantAnalysis(reg_param=0.1)),

        ('gnb',  GaussianNB(var_smoothing=1e-8)),
    ]

    # Method 14: Voting Ensemble
    classifiers['Voting Ensemble'] = Pipeline([
        ('scaler', RobustScaler()),
        ('clf', VotingClassifier(
            estimators=base_estimators, voting='soft',
            weights=[2, 2, 2, 1, 1]))
    ])

    # Method 15: Stacking Ensemble — meta-learner
    classifiers['Stacking Ensemble'] = Pipeline([
        ('scaler', RobustScaler()),
        ('clf', StackingClassifier(
            estimators=base_estimators,
            final_estimator=LogisticRegression(C=1.0, max_iter=500, random_state=RANDOM_STATE),
            cv=5, passthrough=False))
    ])

    results = [{
        'Method':       'Formula (Report)',
        'Accuracy':     formula_acc_tv,
        'Accuracy_std': 0.0,
        'F1_macro':     formula_f1m_tv,
        'F1_macro_std': 0.0,
        'F1_weighted':  formula_f1w_tv,
        'F1_w_std':     0.0,
    }]
    cms = {'Formula (Report)': cm_formula}

    print("\n" + "="*55)
    print("  ML Classifiers — Cross-Validation on Train+Val (5-Fold)")
    print("    Test Set is held out and will not be used until the comparison ends")
    print("="*55)

    best_score   = 0.0
    best_name    = None
    best_clf_obj = None

    for name, clf in classifiers.items():
        print(f"\n  ▸ {name}...")
        scores = evaluate_classifier(name, clf, X_aug, y_aug)
        results.append({'Method': name, **scores})
        cm = full_report(name, clf, X_aug, y_aug)
        cms[name] = cm
        print(f"    CV Accuracy (Train+Val) : {scores['Accuracy']:.4f} ± {scores['Accuracy_std']:.4f}")
        print(f"    CV F1 Macro (Train+Val) : {scores['F1_macro']:.4f} ± {scores['F1_macro_std']:.4f}")

        combined = (scores['Accuracy'] + scores['F1_macro']) / 2
        if combined > best_score and scores['Accuracy'] >= 0.7 and scores['F1_macro'] >= 0.7:
            best_score   = combined
            best_name    = name
            best_clf_obj = clf

    # ── Results Table (Cross-Validation) ──
    results_df = pd.DataFrame(results)
    print("\n" + "="*55)
    print("  Comparison Table — Cross-Validation on Train+Val")
    print("="*55)
    print(results_df[['Method', 'Accuracy', 'F1_macro', 'F1_weighted']].to_string(
        index=False, float_format='{:.4f}'.format))

    best_idx = results_df['Accuracy'].idxmax()
    print(f"\n  Best model (Cross-Val) : {results_df.loc[best_idx,'Method']}")
    print(f"     CV Accuracy          : {results_df.loc[best_idx,'Accuracy']:.4f}")
    print(f"     CV F1 Macro          : {results_df.loc[best_idx,'F1_macro']:.4f}")


    print("\n" + "="*55)
    print("  Final Evaluation on the Independent Test Set")
    print("  (happens only once — after all comparisons are complete)")
    print("="*55)

    y_test_orig = int_to_label(y_test)
    formula_preds_test = np.array([formula_predict(x) for x in X_test])
    test_acc_formula   = accuracy_score(y_test_orig, formula_preds_test)
    test_f1m_formula   = f1_score(y_test_orig, formula_preds_test, average='macro', zero_division=0)
    print(f"\n  Formula — Test Set:")
    print(f"    Test Accuracy : {test_acc_formula:.4f}")
    print(f"    Test F1 Macro : {test_f1m_formula:.4f}")
    print(classification_report(y_test_orig, formula_preds_test,
                                 target_names=LABEL_NAMES, zero_division=0))

    label_map_orig = {-1: 'Low (-1)', 0: 'Medium (0)', 1: 'High (1)'}
    correct_f, wrong_f = [], []
    for i in range(len(y_test_orig)):
        actual    = label_map_orig[y_test_orig[i]]
        predicted = label_map_orig[formula_preds_test[i]]
        fname     = fnames_test[i] if i < len(fnames_test) else f"sample_{i}"
        if y_test_orig[i] == formula_preds_test[i]:
            correct_f.append((fname, actual))
        else:
            wrong_f.append((fname, actual, predicted))

    print(f"\n{'='*55}")


    if best_clf_obj is not None:
        final_name = best_name
        final_clf  = best_clf_obj
    else:
        best_idx2  = results_df[results_df['Method'] != 'Formula (Report)']['Accuracy'].idxmax()
        final_name = results_df.loc[best_idx2, 'Method']
        final_clf  = classifiers[final_name]
        print(f"\n    No model reached 0.7 — the best available model will be evaluated: {final_name}")

    test_acc, test_f1m, test_f1w, cm_test = final_test_evaluation(
        final_name, final_clf, X_aug, y_aug, X_test, y_test,
        fnames_test=fnames_test
    )

    # ── Save Results CSV ──
    results_df.to_csv('module3_results.csv', index=False, float_format='%.4f')
    print("\n  Results saved to module3_results.csv ")

    # ── Save Best Model ──
    final_clf.fit(X_aug, y_aug)
    with open(BEST_CLF_PATH, 'wb') as f:
        pickle.dump({
            'clf':           final_clf,
            'threshold':     THRESHOLD,
            'feature_names': FEATURE_NAMES,
            'best_method':   final_name,
            'cv_score':      best_score,
            'test_accuracy': test_acc,
            'test_f1_macro': test_f1m,
            # Save the split sizes for documentation purposes
            'split_info': {
                'test_size':      TEST_SIZE,
                'n_trainval':     len(X_trainval),
                'n_test':         len(X_test),
                'random_state':   RANDOM_STATE,
            }
        }, f)
    print(f"\n  Best model saved: {final_name} → {BEST_CLF_PATH} ")
    print(f"  Test Accuracy: {test_acc:.4f} | Test F1 Macro: {test_f1m:.4f}")

    # ── Feature Importance (Random Forest + Extra Trees) ──
    print("\n  Feature Importance (Random Forest):")
    rf_pipe = classifiers['Random Forest']
    rf_pipe.fit(X_aug, y_aug)
    importances_rf = rf_pipe.named_steps['clf'].feature_importances_

    print("\n  Feature Importance (Extra Trees):")
    et_pipe = classifiers['Extra Trees']
    et_pipe.fit(X_aug, y_aug)
    importances_et = et_pipe.named_steps['clf'].feature_importances_

    importances_avg = (importances_rf + importances_et) / 2
    print("\n  Feature Importance (average of RF + ET):")
    for fname2, imp in sorted(zip(FEATURE_NAMES, importances_avg), key=lambda x: -x[1]):
        bar = '█' * int(imp * 50)
        print(f"    {fname2:<28} {imp:.4f}  {bar}")

    # ── Plots ──
    print("\n  Generating plots...")
    plot_results(results_df)

    # Confusion matrices:
    cms['Best Model — Test Set'] = cm_test

    n_plots = min(len(cms), 6)
    cols = 3; rows = (n_plots + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
    axes = axes.flatten()
    for ax, (name2, cm) in zip(axes, list(cms.items())[:n_plots]):
        plot_confusion_matrix(cm, name2, ax)
    for ax in axes[n_plots:]:
        ax.set_visible(False)
    plt.suptitle('Confusion Matrices — All Methods\n(Train+Val) + Best Model on Test Set',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('module3_confusion_matrices.png', dpi=150, bbox_inches='tight')
    print("  Confusion matrices saved ")

    print("\n" + "="*55)
    print("  Module III Complete ")
    print(f"  Train+Val: {len(X_trainval)} | Test: {len(X_test)} | SMOTE→{len(X_aug)}")
    print(f"  Best Method (CV)  : {final_name}")
    print(f"  Final Test Acc    : {test_acc:.4f}")
    print(f"  Final Test F1 Mac : {test_f1m:.4f}")
    print("="*55)

In [20]:
main()

Device: cuda

[1] Loading trained ConformerBiLSTM checkpoint...
    Model loaded  (checkpoint epoch=39, val F1=0.6803)

[2] Loading ratings.csv...
    Total entries: 597
    Label distribution:
label
 0    241
-1    195
 1    161

[3] Extracting features from all audio files...

Processing 597 audio files...
  50 processed...
  100 processed...
  150 processed...
  200 processed...
  250 processed...
  300 processed...
  350 processed...
  400 processed...
  450 processed...
  500 processed...
  550 processed...
  Done — 597 files processed, 0 skipped

    Feature matrix shape: (597, 21)
    Labels distribution: {np.int32(-1): np.int64(195), np.int32(0): np.int64(241), np.int32(1): np.int64(161)}

    Feature Statistics:
       breathing_rate  mean_breath_dur  std_breath_dur  pause_regularity  breath_ratio  mean_prob  max_prob  num_segments  rms_mean  rms_std  rms_max  zcr_mean  zcr_std  spec_centroid_mean  spec_centroid_std  spec_bw_mean  spec_rolloff_mean  mfcc1_mean  mfcc2_mean  mfc